# Capital Allocation III

The ideas are in the slides. Here we race the bet-sizing rules, and measure how each one is exposed to what it
estimates.

**The plan**

1. 🎯 "Is this out of sample?": your turn
2. The menu: six rules, the race, and the bootstrap one input at a time. Then back to the slides
3. 🎯 Challenge (homework): audit the book

📑 **Slides for this class:** [Capital Allocation III (PDF)](https://amoreira2.github.io/UG54/slides/capital_allocation_3.pdf)

<iframe src="../../slides/capital_allocation_3.pdf" width="100%" height="520" style="border: 1px solid #ddd;"></iframe>

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [11, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

ff = pd.read_csv(f"{BASE}/ff_monthly.csv", index_col=0, parse_dates=True)   # Ken French, monthly
L  = pd.read_parquet(f"{BASE}/longshort_29.parquet")                        # our 29 long-shorts

F   = ff[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']].dropna()            # the six factors, excess returns
mkt = ff.loc[L.index, 'Mkt-RF']                                             # the market, on the 29's months

sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)
gross1 = lambda w: w / np.abs(w).sum()                                      # scale weights so |w| adds to one

print(f"six factors     {F.index[0]:%Y-%m} to {F.index[-1]:%Y-%m}, {len(F)} months")
print(f"29 long-shorts  {L.index[0]:%Y-%m} to {L.index[-1]:%Y-%m}, {len(L)} months")

---

## 1. 🎯 "Is this out of sample?"

Race mean-variance against 1/N on the six factors. The rule for mean-variance:

- each month $t$, estimate $\mu$ and $\Sigma$ on an expanding window of history
- use `gross1` on $\Sigma^{-1}\mu$ so the absolute weights add to one
- earn month $t$'s return on those weights
- start after 120 months, so the first estimate has ten years of data

Use AI. Call the results `mv_oos` and `eq_oos`: one return per month, over the same months.

Then read the code it gave you, line by line. Which months went into the estimate that earned month $t$?

In [ ]:
# === YOUR TURN ===
MY_PROMPT = """
                                    ← write your prompt here
"""

# ---- paste the AI's code below ----

mv_oos = ____
eq_oos = ____

In [ ]:
#@title 🔒 Check: one index
honest, leak = [], []
for t in range(120, len(F)):
    past = F.iloc[:t]                                                   # strictly before month t
    honest.append(F.iloc[t].values @ gross1(np.linalg.solve(past.cov(), past.mean())))
    peek = F.iloc[:t + 1]                                               # includes month t
    leak.append(F.iloc[t].values @ gross1(np.linalg.solve(peek.cov(), peek.mean())))
honest, leak = [pd.Series(x, index=F.index[120:]) for x in (honest, leak)]
equal = F.iloc[120:].mean(axis=1)

print(f"mean-variance, F.iloc[:t]     Sharpe {sharpe(honest):.2f}")
print(f"mean-variance, F.iloc[:t+1]   Sharpe {sharpe(leak):.2f}   <- one index")
print(f"1/N                           Sharpe {sharpe(equal):.2f}\n")

try:
    print(f"yours: mean-variance {sharpe(mv_oos):.2f}, 1/N {sharpe(eq_oos):.2f}, "
          f"correlation with the honest version {mv_oos.corr(honest):.3f}")
except Exception as e:
    print("yours: not built yet:", type(e).__name__)

- Which line did your AI write? Did it say?
- One month of peeking out of hundreds. Why does it matter so much? Think about what the weights do with month $t$
  if they have seen month $t$.
- Which conclusion would you have reported with the wrong index?

---

## 2. The menu

The six rules from the slides. Each one replaces some estimates with a constant, and still uses the rest:

In [ ]:
RULES = {'mean-variance':   lambda m, C: np.linalg.solve(C, m),                      # means, vols, correlations
         'no correlations': lambda m, C: m / np.diag(C),                             # means, vols
         'same Sharpe':     lambda m, C: np.linalg.solve(C, np.sqrt(np.diag(C))),    # vols, correlations
         'same mean':       lambda m, C: np.linalg.solve(C, np.ones(len(m))),        # vols, correlations
         'risk parity':     lambda m, C: 1 / np.sqrt(np.diag(C)),                    # vols
         '1/N':             lambda m, C: np.ones(len(m))}                            # nothing

# the race: expanding window, strictly before the month earned
race = {k: [] for k in RULES}
for t in range(120, len(F)):
    past = F.iloc[:t]
    for k, f in RULES.items():
        race[k].append(F.iloc[t].values @ gross1(f(past.mean().values, past.cov().values)))
race = pd.DataFrame(race, index=F.index[120:])
print(race.apply(sharpe).round(2).to_string())

In [ ]:
# the bootstrap, one input at a time. Resample the months. From each draw take new means, new volatilities and new
# correlations, and feed in one kind, with the other two at their full-sample values. Then all three together.
HISTORY = len(F)                                    # months in each resampled history. Try 120: ten years of data

split = lambda X: (X.mean(0), X.std(0, ddof=1), np.corrcoef(X.T))
glue  = lambda vol, corr: np.outer(vol, vol) * corr

m0, v0, c0 = split(F.values)
S0 = glue(v0, c0)
best = np.sqrt(m0 @ np.linalg.solve(S0, m0))
INPUTS = ['means', 'vols', 'corrs', 'all']

rng = np.random.default_rng(0)
draws = {k: {i: [] for i in INPUTS} for k in RULES}
for _ in range(1000):
    X = F.values[rng.integers(0, len(F), HISTORY)]
    m1, v1, c1 = split(X)
    for k, f in RULES.items():
        draws[k]['means'].append(gross1(f(m1, S0)))
        draws[k]['vols'].append(gross1(f(m0, glue(v1, c0))))
        draws[k]['corrs'].append(gross1(f(m0, glue(v0, c1))))
        draws[k]['all'].append(gross1(f(m1, glue(v1, c1))))

# how much the weights move, and what that costs, scored against the full-sample inputs as if they were the truth
var_w, lost = {}, {}
for k, f in RULES.items():
    w0 = gross1(f(m0, S0))
    sr0 = w0 @ m0 / np.sqrt(w0 @ S0 @ w0)
    var_w[k], lost[k] = {}, {}
    for i in INPUTS:
        W = np.array(draws[k][i])
        var_w[k][i] = W.var(axis=0).mean() * 1e4
        lost[k][i] = (sr0 - np.mean(W @ m0 / np.sqrt(np.sum((W @ S0) * W, axis=1)))) / best * 100
print('average variance of a weight, times 10,000')
print(pd.DataFrame(var_w).T.round(1).to_string())
print('\nSharpe ratio lost, % of the best')
print(pd.DataFrame(lost).T.round(1).to_string())

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for ax, i in zip(axes, INPUTS):
    ax.boxplot(np.array(draws['mean-variance'][i]), showfliers=False)
    ax.set_xticklabels(F.columns, fontsize=8); ax.set_title('mean-variance: ' + i); ax.axhline(0, color='k', lw=0.5)
plt.show()

- For mean-variance, which input moves the weights most? Which one barely matters?
- Read down the table: which rule is exposed to what? Compare with the list of what each one still estimates.
- Set `HISTORY = 120` and rerun. What happens to the Sharpe ratio lost, and to which rules?
- These scores treat the full-sample means as the truth. Whom does that flatter?
- Which rule won the race? Would you bet that it wins on any set of assets? **Bring this back to the slides.**

---

## ⏸ Back to the slides

The challenge below is homework. It uses the second half of the slides.

---

## 🎯 Challenge: audit the book

*Homework — due before the next class.*

You inherit the 29-strategy book you built in *Capital Allocation II* and are asked to defend how it is built.

### Q1 — The control

Run a control: throw the alphas away. Compare two rules, out of sample, on an expanding window with a 120-month
burn-in:

- **the alpha book**: $w_i \propto \alpha_i / \sigma^2_{\varepsilon,i}$
- **no alpha**: $w_i \propto 1 / \sigma^2_{\varepsilon,i}$

Each month $t$, **using only the months before $t$**:

1. regress each strategy on the market to get $\alpha_i$, $\beta_i$ and $\sigma_{\varepsilon,i}$
2. form the weights and apply `gross1`
3. earn the hedged return in month $t$: $r_{i,t} - \beta_i\, r_{m,t}$, with the $\beta_i$ from step 1

> **⚠️ Step 3 is where this goes wrong.** Hedging with a beta from the whole sample looks ahead, and it changes the
> answer a lot.

Report the Sharpe ratio of the no-alpha rule minus the Sharpe ratio of the alpha book. A positive number means the
alphas cost you.

> **📌 Required variable names:**
> ```python
> alpha_cost = ____   # SR(no alpha) - SR(alpha book), out of sample
> ```

In [ ]:
# Your work here


alpha_cost = ____

print(f"discarding the alphas is worth {alpha_cost:+.3f} of Sharpe ratio")

### Q2 — The right amount of shrinkage

On the **six factors**, same expanding window and burn-in, shrink the covariance toward the scaled identity:

$$(1-a)\,\Sigma \;+\; a\,\frac{\operatorname{tr}\Sigma}{6}\, I$$

Search $a$ over `[0, 0.1, 0.2, ..., 1.0]`, using `gross1` on the weights, and report the $a$ with the highest
out-of-sample Sharpe ratio.

> **📌 Required variable names:**
> ```python
> best_a = ____   # the shrinkage intensity, a number in [0, 1]
> ```

In [ ]:
# Your work here


best_a = ____

print(f"best shrinkage intensity: {best_a:.1f}")

### Q3 — Does the shrinkage choice survive?

Q2 picked $a$ knowing the whole sample. Now do what a real manager must: pick it **without** seeing the future.

Keep the out-of-sample return series for each $a$ from Q2. Each month, choose the $a$ whose series has the highest
Sharpe ratio over the months so far, and use that $a$ for the next month's return. Use $a = 0$ for the first 60
out-of-sample months, until there is a track record to choose from.

Report the Sharpe ratio of that live procedure over all the out-of-sample months, and compare it with Q2 and with
plain mean-variance from class.

> **📌 Required variable names:**
> ```python
> live_shrinkage_sr = ____   # out-of-sample Sharpe when a is chosen live
> ```

In [ ]:
# Your work here


live_shrinkage_sr = ____

print(f"choosing a live: {live_shrinkage_sr:.3f}")

### Q4 — The memo

> **📝 Your task — maximum eight sentences.**
>
> Your firm runs the 29-strategy book. The head of research proposes replacing $\alpha/\sigma^2_\varepsilon$ with
> plain $1/\sigma^2_\varepsilon$, throwing away the alpha estimates entirely, on the evidence in Q1.
>
> Say whether you agree and why, and be precise about what Q1 does and does not establish. How would you put a
> standard error on it? Then use Q2 and Q3 together to answer the harder question: **you found a shrinkage intensity
> that improves things, and then found that choosing it honestly gives the improvement back. What, if anything, have
> you learned?** Finish by naming the one piece of evidence that would most change your mind about the alphas.

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["alpha_cost", "best_a", "live_shrinkage_sr", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L13_CapitalAllocationIII_v2_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")